In [7]:
import pandas as pd
import pysam
import re
import numpy as np
from itertools import combinations
from glob import glob
import re
import networkx as nx
import ipynbname
pd.set_option("display.max_columns", None)

In [8]:
folder_path = str(ipynbname.path().parent) + "/../"
print(folder_path)

/Genomics/argo/users/yz8667/Autism_network_integration/scripts/GitHub/analysis/../


# General

In [9]:
def format_mpc(x):
    output = []
    for item in str(x).split(";"):
        if item == ".":
            output.append(np.nan)
        else:
            output.append(float(item))
    return np.nanmax(output)

In [10]:
def dataframe_to_vcf(
    df,
    variant_col="Variant",
    output_vcf="output.vcf",
    info_cols=None,
    remove_chr_prefix=False,
    keep_missing_as_dot=True
):
    df = df.copy()

    # Split CHR:POS:REF:ALT
    split_var = df[variant_col].astype(str).str.split(":", expand=True)
    split_var.columns = ["CHROM", "POS", "REF", "ALT"]

    if remove_chr_prefix:
        split_var["CHROM"] = split_var["CHROM"].str.replace("^chr", "", regex=True)

    # Decide which columns to keep in INFO
    if info_cols is None:
        info_cols = [c for c in df.columns if c != variant_col]

    def clean_info_key(x):
        return re.sub(r"[^A-Za-z0-9_.-]", "_", str(x))

    def clean_info_value(x):
        if pd.isna(x):
            return "." if keep_missing_as_dot else None
        x = str(x)
        x = x.replace(";", ",").replace("=", ":").replace("\t", "_").replace(" ", "_")
        return x

    def chrom_to_int(chrom):
        chrom = str(chrom).replace("chr", "")
        if chrom == "X":
            return 23
        elif chrom == "Y":
            return 24
        elif chrom in ["M", "MT"]:
            return 25
        else:
            return int(chrom)

    info_col = []

    for _, row in df.iterrows():
        info_items = []

        for c in info_cols:
            val = clean_info_value(row[c])

            if val is None:
                continue

            info_items.append(f"{clean_info_key(c)}={val}")

        info_col.append(";".join(info_items) if info_items else ".")

    vcf = pd.DataFrame({
        "#CHROM": split_var["CHROM"],
        "POS": split_var["POS"],
        "ID": ".",
        "REF": split_var["REF"],
        "ALT": split_var["ALT"],
        "QUAL": ".",
        "FILTER": ".",
        "INFO": info_col
    })
    
    # sort by CHROM and POS
    vcf["_chrom_sort"] = vcf["#CHROM"].apply(chrom_to_int)
    vcf["POS"] = vcf["POS"].astype(int)
    vcf = vcf.sort_values(["_chrom_sort", "POS"]).drop(columns="_chrom_sort")

    with open(output_vcf, "w") as f:
        f.write("##fileformat=VCFv4.2\n")
        for c in info_cols:
            key = clean_info_key(c)
            f.write(
                f'##INFO=<ID={key},Number=1,Type=String,Description="{key} from input table">\n'
            )
        vcf.to_csv(f, sep="\t", index=False)

    print(f"saved sorted VCF to: {output_vcf}")
    return vcf

In [11]:
def parse_vep(
    vep_file,
    output_file=None,
    return_df=True
):
    """
    Parse VEP output into a clean tab-separated annotation table.

    Parameters
    ----------
    vep_file : str
        Path to VEP output file

    output_file : str or None
        If provided, save parsed table to TSV

    return_df : bool
        Whether to return pandas DataFrame

    Returns
    -------
    pandas.DataFrame
    """

    # --------------------------------------------------
    # Get header
    # --------------------------------------------------
    with open(vep_file) as f:
        for line in f:
            if line.startswith("#Uploaded_variation"):
                header = line.lstrip("#").rstrip("\n").split("\t")
                break

    # --------------------------------------------------
    # Read main VEP table
    # --------------------------------------------------
    df = pd.read_csv(
        vep_file,
        sep="\t",
        comment="#",
        names=header,
        dtype=str
    )

    # --------------------------------------------------
    # Parse Extra column
    # --------------------------------------------------
    def parse_extra(extra):

        if pd.isna(extra) or extra == "-":
            return {}

        out = {}

        for item in extra.split(";"):

            if "=" in item:
                k, v = item.split("=", 1)
                out[k] = v

            else:
                out[item] = True

        return out

    extra_df = df["Extra"].apply(parse_extra).apply(pd.Series)

    # --------------------------------------------------
    # Combine
    # --------------------------------------------------
    anno = pd.concat(
        [df.drop(columns=["Extra"]), extra_df],
        axis=1
    )

    # --------------------------------------------------
    # Save
    # --------------------------------------------------
    if output_file is not None:

        anno.to_csv(
            output_file,
            sep="\t",
            index=False
        )

        print(f"saved parsed annotations to: {output_file}")

    if return_df:
        return anno

In [12]:
def parse_simple_vcf(vcf_file, output_file=None, return_df=True):
    """
    Parse a simple VCF file and expand INFO fields into columns.
    """

    # Read VCF main table
    df = pd.read_csv(
        vcf_file,
        sep="\t",
        comment="#",
        header=None,
        names=["CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO"],
        dtype=str
    )

    # Parse INFO column
    def parse_info(info):
        info_dict = {}
        if pd.isna(info) or info == ".":
            return info_dict

        for item in info.split(";"):
            if "=" in item:
                key, value = item.split("=", 1)
                info_dict[key] = value
            else:
                info_dict[item] = True

        return info_dict

    info_df = df["INFO"].apply(parse_info).apply(pd.Series)

    # Merge VCF core columns + parsed INFO columns
    df_out = pd.concat(
        [df.drop(columns=["INFO"]), info_df],
        axis=1
    )

    # Optional: convert POS to integer
    df_out["POS"] = df_out["POS"].astype(int)

    # Optional: replace "." with NA
    df_out = df_out.replace(".", pd.NA)

    if output_file is not None:
        df_out.to_csv(output_file, sep="\t", index=False)

    if return_df:
        return df_out

In [13]:
def damaging_flag(row, metrics):
    for metric in metrics.split(","):
        if metric == "am_class" and row[metric] == "likely_pathogenic":
            return True
        if metric == "REVEL" and row[metric] >= 0.5:
            return True
        if metric == "MPC" and row[metric] >= 2:
            return True
    return False

In [14]:
def get_disrupted_ppi(row, mut_res_list):
    if type(row["P1_IRES"]) == str and set(row["P1_IRES"].split(",")).intersection(mut_res_list) != set():
        return True
    if type(row["P2_IRES"]) == str and set(row["P2_IRES"].split(",")).intersection(mut_res_list) != set():
        return True
    return False

In [15]:
def parse_netflow3d_output(dr, file_tag, size_cutoff):
    G = nx.read_graphml(f"{dr}{file_tag}/initial_state.graphml.gz")
    df = pd.read_csv(f"{dr}{file_tag}_subnetworks.txt", sep="\t")
    df_netflow3d = df[df["Subnetwork_size"] > size_cutoff]
    netflow3d_prots = []
    for x in df_netflow3d["Subnetwork_genes"]:
        prots = x.split(",")
        netflow3d_prots.extend(prots)
    netflow3d_prots = set(netflow3d_prots)
    print(f"Number of proteins in NetFlow3D subnetworks: {len(netflow3d_prots)}")
    print(f"Number of subnetworks: {len(df_netflow3d)}")
    return G, df_netflow3d, netflow3d_prots

In [16]:
def get_jaccard(subnetwork_df, pathway_df, group_name, df_idmap, output=None):
    pathway_sets = {
        row["Biological_process"]: set(row["Genes"].split(","))
        for _, row in pathway_df.iterrows()
    }

    total_rows = []
    for subnetwork in subnetwork_df["Subnetwork_genes"]:
        uniprots = set(subnetwork.split(","))
        genes = set(df_idmap[df_idmap["UniProt"].isin(uniprots)]["ID"])

        for pathway, pathway_genes in pathway_sets.items():
            intersect = genes & pathway_genes
            union = genes | pathway_genes
            jaccard = len(intersect) / len(union) if union else 0

            total_rows.append({
                "Subnetwork_genes": ",".join(genes),
                "GO_Pathway": pathway,
                "Jaccard": jaccard,
                "Group": group_name,
                "Overlapped_elements": ",".join(intersect),
                "Numerator": len(intersect),
                "Denominator": len(union)
            })

    df_total = pd.DataFrame(total_rows)

    idx = df_total.groupby("Subnetwork_genes")["Jaccard"].idxmax()
    df_final = df_total.loc[idx].reset_index(drop=True)

    if output is not None:
        df_final.to_csv(f"{output}{group_name}_maxJaccard.txt", sep="\t", index=False)
        df_total.to_csv(f"{output}{group_name}_totalJaccard.txt", sep="\t", index=False)

    return df_final

In [17]:
# load canonical UniProts
fasta_dict = {}
with open(f"{folder_path}/src/NetFlow3D/metadata/UP000005640_9606.fasta") as f:
    header = None
    seq_chunks = []

    for line in f:
        line = line.strip()
        if line.startswith(">"):
            if header is not None:
                fasta_dict[header] = "".join(seq_chunks)

            # parse header: >tr|A0A075B6V1|A0A075B6V1_HUMAN ...
            parts = line[1:].split("|")
            header = parts[1]
            seq_chunks = []
        else:
            seq_chunks.append(line)

    # last sequence
    if header is not None:
        fasta_dict[header] = "".join(seq_chunks)
len(fasta_dict)

20596

In [18]:
# load id mapping file
df_idmap = pd.read_csv(f"{folder_path}/src/NetFlow3D/metadata/HUMAN_9606_idmapping.dat.gz", sep="\t", header=None, names=["UniProt", "ID_type", "ID"])
df_idmap["UniProt"] = df_idmap["UniProt"].apply(lambda x: x.split("-")[0])
df_idmap["ID"] = df_idmap["ID"].apply(lambda x: x.split(".")[0])
df_enstmap = df_idmap[df_idmap["ID_type"] == "Ensembl_TRS"].copy()
df_enspmap = df_idmap[df_idmap["ID_type"] == "Ensembl_PRO"].copy()
df_ensgmap = df_idmap[df_idmap["ID_type"] == "Ensembl"].copy()
df_idmap = df_idmap[df_idmap["ID_type"] == "Gene_Name"]
df_idmap_slim = df_idmap[df_idmap["UniProt"].isin(fasta_dict.keys())]
display(df_idmap)
display(df_idmap_slim)

,UniProt,ID_type,ID
1,P31946,Gene_Name,YWHAB
118,P62258,Gene_Name,YWHAE
246,Q04917,Gene_Name,YWHAH
336,P61981,Gene_Name,YWHAG
462,P31947,Gene_Name,SFN
...,...,...,...
6198707,A0A678ZI47,Gene_Name,HLA-DRB1
6198717,A0A678ZTH6,Gene_Name,HLA-DQB1
6198728,A0A1C8DU67,Gene_Name,HLA-DRB1
6198740,A0A1D8MAN8,Gene_Name,HLA-A


,UniProt,ID_type,ID
1,P31946,Gene_Name,YWHAB
118,P62258,Gene_Name,YWHAE
246,Q04917,Gene_Name,YWHAH
336,P61981,Gene_Name,YWHAG
462,P31947,Gene_Name,SFN
...,...,...,...
5924608,A0A0J9YVS3,Gene_Name,IGHJ4
6024285,A0A0J9YW14,Gene_Name,IGHJ3
6026795,A0A0G2JNF4,Gene_Name,KIR2DP1
6062812,A0A140TA62,Gene_Name,LOC100653049


In [19]:
# load GO biological processes
GO = pd.read_csv(f"{folder_path}/ref/GO_biological_processes.txt", sep = "\t")

# Compile raw variants

## load UCSF de novo variants

In [40]:
dr = "/Genomics/argo/users/yz8667/Autism_network_integration/raw_data/UCSF_raw/"
df1 = pd.read_csv(f"{dr}UCSF_20211001_dnms_WGS_hg38.tsv", sep = "\t")
df1

/tmp/ipykernel_76360/710857299.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv(f"{dr}UCSF_20211001_dnms_WGS_hg38.tsv", sep = "\t")


,CHROM,POS,ID,REF,ALT,TYPE,LEN,FAM,SAMPLE,STATUS,BATCH
0,chr10,7186588,chr10:7186588:T:C,T,C,SNV,0,12071,SSC03607,case,SSC_WGS_9209
1,chr10,20430290,chr10:20430290:C:T,C,T,SNV,0,12071,SSC03607,case,SSC_WGS_9209
2,chr10,22016032,chr10:22016032:G:A,G,A,SNV,0,12071,SSC03607,case,SSC_WGS_9209
3,chr10,109434004,chr10:109434004:A:G,A,G,SNV,0,12071,SSC03607,case,SSC_WGS_9209
4,chr11,26613411,chr11:26613411:A:C,A,C,SNV,0,12071,SSC03607,case,SSC_WGS_9209
...,...,...,...,...,...,...,...,...,...,...,...
526329,chr10,33634582,chr10:33634582:ATGT:A,ATGT,A,Indel,3,SF0248880,SP0248880,case,SPARK_WGS_3
526330,chr11,28855264,chr11:28855264:AG:A,AG,A,Indel,1,SF0248880,SP0248880,case,SPARK_WGS_3
526331,chr2,152807977,chr2:152807977:AC:A,AC,A,Indel,1,SF0248880,SP0248880,case,SPARK_WGS_3
526332,chr6,68321063,chr6:68321063:TTTA:T,TTTA,T,Indel,3,SF0248880,SP0248880,case,SPARK_WGS_3


In [41]:
df1.groupby("TYPE").size()

TYPE
Indel     44233
SNV      482101
dtype: int64

In [42]:
df1["Variant"] = df1["ID"].apply(lambda x: x[3:])
df1 = df1[["Variant", "TYPE", "SAMPLE", "STATUS"]]
df1["Source"] = "UCSF"
df1.head()

/tmp/ipykernel_76360/508363256.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["Source"] = "UCSF"


,Variant,TYPE,SAMPLE,STATUS,Source
0,10:7186588:T:C,SNV,SSC03607,case,UCSF
1,10:20430290:C:T,SNV,SSC03607,case,UCSF
2,10:22016032:G:A,SNV,SSC03607,case,UCSF
3,10:109434004:A:G,SNV,SSC03607,case,UCSF
4,11:26613411:A:C,SNV,SSC03607,case,UCSF


In [43]:
# grab yutong's MPC annotation
dr = f"{folder_path}data/analysis/Disrupted_PPI/"
bcf = pysam.VariantFile(dr + "UCSF_20211001_dnms_WGS_hg38.includemultiplex.codingonly.filtered.vep.with_asdinfo.bcf")
csq_desc = bcf.header.info["CSQ"].description
fields = csq_desc.split("Format: ")[1].split("|")

df_output = []
for rec in bcf:
    try:
        mpc_score = rec.info["MPC"]
    except:
        continue
    for i,allele in enumerate(rec.info["CSQ"]):
        df_output.append(dict(zip(["chrom", "pos", "ref", "STATUS", "MPC"] + fields, [rec.chrom, rec.pos, rec.ref, rec.info["STATUS"], mpc_score] + allele.split("|"))))
df_output = pd.DataFrame(df_output)
df_output["Variant"] = df_output.apply(lambda x: f"{x['chrom']}:{x['pos']}:{x['ref']}:{x['Allele']}", axis = 1)
df_output = df_output[df_output["Consequence"].apply(lambda x: "missense_variant" in x.split("&"))]
df_output

,chrom,pos,ref,STATUS,MPC,Allele,Consequence,Gene,Feature_type,Feature,SIFT,PolyPhen,REVEL,CADD_PHRED,CADD_RAW,phyloP100way_max,phastCons100way_max,gnomADg_v3_AF,gnomADg_AF,SYMBOL,HGVSc,HGVSp,Protein_position,Amino_acids,Codons,Variant
1,1,972320,G,case,0.013650,A,missense_variant,ENSG00000187583,Transcript,ENST00000379407,tolerated(0.06),possibly_damaging(0.464),0.189,19.65,2.652424,0.419999986886978,0.00100000004749745,,1.314e-05,PLEKHN1,,,312,A/T,Gcc/Acc,1:972320:G:A
2,1,972320,G,case,0.013650,A,missense_variant,ENSG00000187583,Transcript,ENST00000379409,tolerated(0.06),benign(0.25),0.189,19.65,2.652424,0.419999986886978,0.00100000004749745,,1.314e-05,PLEKHN1,,,352,A/T,Gcc/Acc,1:972320:G:A
3,1,972320,G,case,0.013650,A,missense_variant,ENSG00000187583,Transcript,ENST00000379410,tolerated(0.08),benign(0.32),0.189,19.65,2.652424,0.419999986886978,0.00100000004749745,,1.314e-05,PLEKHN1,,,300,A/T,Gcc/Acc,1:972320:G:A
9,1,972320,G,case,0.013650,A,missense_variant,ENSG00000187583,Transcript,ENST00000880608,tolerated(0.07),benign(0.157),0.189,19.65,2.652424,0.419999986886978,0.00100000004749745,,1.314e-05,PLEKHN1,,,312,A/T,Gcc/Acc,1:972320:G:A
10,1,972320,G,case,0.013650,A,missense_variant,ENSG00000187583,Transcript,ENST00000880609,,,0.189,19.65,2.652424,0.419999986886978,0.00100000004749745,,1.314e-05,PLEKHN1,,,312,A/T,Gcc/Acc,1:972320:G:A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93609,22,50419410,C,case,0.455965,G,missense_variant,ENSG00000100239,Transcript,ENST00000966989,deleterious_low_confidence(0),benign(0.267),0.084,21.4,3.087467,2.53099989891052,1,,,PPP6R2,,,265,L/V,Ctc/Gtc,22:50419410:C:G
93610,22,50419410,C,case,0.455965,G,missense_variant,ENSG00000100239,Transcript,ENST00000966990,,,0.084,21.4,3.087467,2.53099989891052,1,,,PPP6R2,,,231,L/V,Ctc/Gtc,22:50419410:C:G
93611,22,50419410,C,case,0.455965,G,missense_variant,ENSG00000100239,Transcript,ENST00000966991,deleterious_low_confidence(0),benign(0.141),0.084,21.4,3.087467,2.53099989891052,1,,,PPP6R2,,,265,L/V,Ctc/Gtc,22:50419410:C:G
93615,22,50482584,G,case,0.063324,T,missense_variant,ENSG00000128165,Transcript,ENST00000395737,tolerated(0.11),benign(0),0.028,0.609,-0.143656,-1.87800002098083,0,,6.565e-06,ADM2,,,43,S/I,aGc/aTc,22:50482584:G:T


In [44]:
df1 = pd.merge(df1, df_output[["Variant", "MPC"]].drop_duplicates(), on = "Variant")
df1

,Variant,TYPE,SAMPLE,STATUS,Source,MPC
0,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419
1,3:123327758:G:A,SNV,SSC04674,case,UCSF,1.435280
2,7:55964634:C:T,SNV,SSC05819,control,UCSF,0.607774
3,3:8994422:C:T,SNV,SSC12506,case,UCSF,1.520430
4,6:32195810:C:G,SNV,SSC12506,case,UCSF,0.894647
...,...,...,...,...,...,...
4881,22:24185252:C:G,SNV,SP0216787,case,UCSF,0.752381
4882,11:48121048:C:G,SNV,SP0253549,case,UCSF,0.456474
4883,14:73264305:G:A,SNV,SP0253549,case,UCSF,0.183579
4884,16:67628392:A:C,SNV,SP0192525,case,UCSF,1.937360


In [45]:
len(df1["MPC"].dropna())

4886

## load Fu de novo variants

In [46]:
df2 = pd.read_excel(f"{folder_path}ref/NIHMS1844621-supplement-Supplementary_Table.xlsx", sheet_name="Supplementary Table 20").dropna(subset = ["Role"])
df2 = df2[df2["isMis"] == 1.0]
df2

,Variant,Sample,Role,Father,Mother,gene,gene_id,Simplified_csq,isIndel,isPTV,isMisB,isMisA,isMis,isSyn,MPC
0,12:9113374:A:G,14397_s1,Sibling,14397_fa,14397_mo,A2M,ENSG00000175899,missense_variant,0.0,0.0,0.0,1.0,1.0,0.0,1.076200
1,12:9077413:A:C,G01-GEA-209-HI,Proband,G01-GEA-209-PA,G01-GEA-209-MA,A2M,ENSG00000175899,missense_variant,0.0,0.0,0.0,0.0,1.0,0.0,0.685040
3,12:8857229:G:A,SSC03765,Proband,SSC03769,SSC03767,A2ML1,ENSG00000166535,missense_variant,0.0,0.0,0.0,0.0,1.0,0.0,0.560810
4,12:8863976:C:T,DEASD_3011_002,Sibling,DEASD_3011_500,DEASD_3011_600,A2ML1,ENSG00000166535,missense_variant,0.0,0.0,0.0,0.0,1.0,0.0,0.130360
6,12:53314298:C:T,08C75732,Proband,08C75733,08C75642,AAAS,ENSG00000094914,missense_variant,0.0,0.0,0.0,0.0,1.0,0.0,0.397820
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35251,19:11723008:G:C,SP0023172,Proband,SP0023222,SP0023146,ZNF823,ENSG00000197933,missense_variant,0.0,0.0,0.0,0.0,1.0,0.0,0.417360
35252,17:34961586:C:T,SP0022147,Proband,SP0028790,SP0022128,ZNF830,ENSG00000198783,missense_variant,0.0,0.0,0.0,0.0,1.0,0.0,0.424820
35255,16:21203187:C:T,SP0001383,Proband,SP0001348,SP0000360,ZP2,ENSG00000103310,missense_variant,0.0,0.0,0.0,0.0,1.0,0.0,0.067960
35256,7:76425079:G:A,SP0003489,Proband,SP0003232,SP0001192,ZP3,ENSG00000188372,missense_variant,0.0,0.0,0.0,0.0,1.0,0.0,0.014326


In [47]:
df2.groupby("isIndel").size()

isIndel
0.0    14958
dtype: int64

In [48]:
df2["TYPE"] = df2["isIndel"].apply(lambda x: "Indel" if x==1 else "SNV")
df2["STATUS"] = df2["Role"].apply(lambda x: "case" if x == "Proband" else "control")
df2 = df2.rename(columns = {"Sample": "SAMPLE"})
df2["Source"] = "Fu"
df2 = df2[["Variant", "TYPE", "SAMPLE", "STATUS", "Source", "MPC"]]
df2

,Variant,TYPE,SAMPLE,STATUS,Source,MPC
0,12:9113374:A:G,SNV,14397_s1,control,Fu,1.076200
1,12:9077413:A:C,SNV,G01-GEA-209-HI,case,Fu,0.685040
3,12:8857229:G:A,SNV,SSC03765,case,Fu,0.560810
4,12:8863976:C:T,SNV,DEASD_3011_002,control,Fu,0.130360
6,12:53314298:C:T,SNV,08C75732,case,Fu,0.397820
...,...,...,...,...,...,...
35251,19:11723008:G:C,SNV,SP0023172,case,Fu,0.417360
35252,17:34961586:C:T,SNV,SP0022147,case,Fu,0.424820
35255,16:21203187:C:T,SNV,SP0001383,case,Fu,0.067960
35256,7:76425079:G:A,SNV,SP0003489,case,Fu,0.014326


## combine and save to vcf

In [49]:
df_concat = pd.concat([df1, df2], axis = 0)
df_concat["MPC"] = df_concat["MPC"].astype(float)
df_concat

,Variant,TYPE,SAMPLE,STATUS,Source,MPC
0,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419
1,3:123327758:G:A,SNV,SSC04674,case,UCSF,1.435280
2,7:55964634:C:T,SNV,SSC05819,control,UCSF,0.607774
3,3:8994422:C:T,SNV,SSC12506,case,UCSF,1.520430
4,6:32195810:C:G,SNV,SSC12506,case,UCSF,0.894647
...,...,...,...,...,...,...
35251,19:11723008:G:C,SNV,SP0023172,case,Fu,0.417360
35252,17:34961586:C:T,SNV,SP0022147,case,Fu,0.424820
35255,16:21203187:C:T,SNV,SP0001383,case,Fu,0.067960
35256,7:76425079:G:A,SNV,SP0003489,case,Fu,0.014326


In [50]:
df_concat["STATUS"].unique()

array(['case', 'control'], dtype=object)

In [51]:
df1["STATUS"].unique()

array(['case', 'control'], dtype=object)

In [52]:
df2["STATUS"].unique()

array(['control', 'case'], dtype=object)

In [53]:
dr = f"{folder_path}data/analysis/Disrupted_PPI/"
df_concat.to_csv(f"{dr}UCSF_Fu_de_novo_variants_hg38.txt", index = False, sep = "\t")

In [ ]:
# convert to vcf for annotation in VEP
dr = f"{folder_path}data/analysis/Disrupted_PPI/"

vcf = dataframe_to_vcf(
    df_concat,
    variant_col="Variant",
    output_vcf=f"{dr}UCSF_Fu_de_novo_variants_hg38.vcf",
    info_cols=["TYPE", "SAMPLE", "STATUS", "Source", "MPC"]
)

vcf.head()

saved sorted VCF to: /Genomics/argo/users/yz8667/Autism_network_integration/data/PPI_interface/UCSF_Fu_de_novo_variants_hg38.vcf


,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
28736,1,939437,.,C,T,.,.,TYPE=SNV;SAMPLE=SP0012309;STATUS=case;Source=F...
6751,1,946422,.,C,T,.,.,TYPE=SNV;SAMPLE=AC01-0062-01;STATUS=case;Sourc...
5128,1,961517,.,C,T,.,.,TYPE=SNV;SAMPLE=400-06-106733;STATUS=control;S...
5129,1,962359,.,T,C,.,.,TYPE=SNV;SAMPLE=DEASD_1049_001;STATUS=case;Sou...
22843,1,962373,.,A,G,.,.,TYPE=SNV;SAMPLE=SP0058285;STATUS=control;Sourc...


In [446]:
vcf.shape[0]

19844

In [447]:
vcf[["#CHROM", "POS", "REF", "ALT"]].drop_duplicates().shape[0]

17789

In [448]:
vcf["INFO"].tolist()[0]

'TYPE=SNV;SAMPLE=SP0012309;STATUS=case;Source=Fu;MPC=0.0071438'

# Run VEP and parse output

In [ ]:
cd {folder_path}analysis
sbatchlog 0_run_VEP.sh {folder_path}data/analysis/Disrupted_PPI/UCSF_Fu_de_novo_variants_hg38.vcf {folder_path}data/analysis/Disrupted_PPI/


In [55]:
# load vep output
dr = f"{folder_path}data/analysis/Disrupted_PPI/"
df_vep = parse_vep(f"{dr}UCSF_Fu_de_novo_variants_hg38.vep.txt",
    output_file=f"{dr}UCSF_Fu_de_novo_variants_hg38.parsed_vep.txt",
    return_df=True
)
df_vep["Variant"] = df_vep["Uploaded_variation"].apply(lambda x: x.replace("_", ":").replace("/", ":"))
df_vep

saved parsed annotations to: /Genomics/argo/users/yz8667/Autism_network_integration/scripts/GitHub/analysis/../data/analysis/Disrupted_PPI/UCSF_Fu_de_novo_variants_hg38.parsed_vep.txt


,Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,Existing_variation,IMPACT,DISTANCE,STRAND,VARIANT_CLASS,SYMBOL,SYMBOL_SOURCE,HGNC_ID,CANONICAL,ENSP,gnomADe_AF,gnomADe_AFR_AF,gnomADe_AMR_AF,gnomADe_ASJ_AF,gnomADe_EAS_AF,gnomADe_FIN_AF,gnomADe_NFE_AF,gnomADe_OTH_AF,gnomADe_SAS_AF,gnomADg_AF,gnomADg_AFR_AF,gnomADg_AMI_AF,gnomADg_AMR_AF,gnomADg_ASJ_AF,gnomADg_EAS_AF,gnomADg_FIN_AF,gnomADg_MID_AF,gnomADg_NFE_AF,gnomADg_OTH_AF,gnomADg_SAS_AF,CLIN_SIG,PHENO,FLAGS,REVEL,am_class,am_pathogenicity,SOMATIC,Variant
0,1_939437_C/T,1:939437,T,ENSG00000188976,ENST00000327044,Transcript,downstream_gene_variant,-,-,-,-,-,rs201898716,MODIFIER,4766,-1,SNV,NOC2L,HGNC,HGNC:24517,YES,ENSP00000317992,0.0003007,0,0.000449,0.0007165,0,5.035e-05,0.0001117,0.0003389,0.001172,0.0001626,2.625e-05,0,0.000144,0.0006169,0,0.0001157,0,0.0001687,0,0.001397,uncertain_significance,1,NaN,NaN,NaN,NaN,NaN,1:939437:C:T
1,1_939437_C/T,1:939437,T,ENSG00000187634,ENST00000341065,Transcript,intron_variant,-,-,-,-,-,rs201898716,MODIFIER,NaN,1,SNV,SAMD11,HGNC,HGNC:28706,NaN,ENSP00000349216,0.0003007,0,0.000449,0.0007165,0,5.035e-05,0.0001117,0.0003389,0.001172,0.0001626,2.625e-05,0,0.000144,0.0006169,0,0.0001157,0,0.0001687,0,0.001397,uncertain_significance,1,cds_start_NF,NaN,NaN,NaN,NaN,1:939437:C:T
2,1_939437_C/T,1:939437,T,ENSG00000187634,ENST00000342066,Transcript,missense_variant,773,683,228,P/L,cCc/cTc,rs201898716,MODERATE,NaN,1,SNV,SAMD11,HGNC,HGNC:28706,NaN,ENSP00000342313,0.0003007,0,0.000449,0.0007165,0,5.035e-05,0.0001117,0.0003389,0.001172,0.0001626,2.625e-05,0,0.000144,0.0006169,0,0.0001157,0,0.0001687,0,0.001397,uncertain_significance,1,NaN,0.080,likely_benign,0.0781,NaN,1:939437:C:T
3,1_939437_C/T,1:939437,T,ENSG00000187634,ENST00000437963,Transcript,downstream_gene_variant,-,-,-,-,-,rs201898716,MODIFIER,3644,1,SNV,SAMD11,HGNC,HGNC:28706,NaN,ENSP00000393181,0.0003007,0,0.000449,0.0007165,0,5.035e-05,0.0001117,0.0003389,0.001172,0.0001626,2.625e-05,0,0.000144,0.0006169,0,0.0001157,0,0.0001687,0,0.001397,uncertain_significance,1,cds_end_NF,NaN,NaN,NaN,NaN,1:939437:C:T
4,1_939437_C/T,1:939437,T,ENSG00000187634,ENST00000455979,Transcript,missense_variant,163,164,55,P/L,cCc/cTc,rs201898716,MODERATE,NaN,1,SNV,SAMD11,HGNC,HGNC:28706,NaN,ENSP00000412228,0.0003007,0,0.000449,0.0007165,0,5.035e-05,0.0001117,0.0003389,0.001172,0.0001626,2.625e-05,0,0.000144,0.0006169,0,0.0001157,0,0.0001687,0,0.001397,uncertain_significance,1,cds_start_NF,NaN,NaN,NaN,NaN,1:939437:C:T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191769,22_50768826_T/A,22:50768826,A,ENSG00000079974,ENST00000482308,Transcript,downstream_gene_variant,-,-,-,-,-,rs1555915879,MODIFIER,701,-1,SNV,RABL2B,HGNC,HGNC:9800,NaN,NaN,3.982e-06,0,0,0,0,0,8.817e-06,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22:50768826:T:A
191770,22_50768826_T/A,22:50768826,A,ENSG00000291064,ENST00000496652,Transcript,"intron_variant,non_coding_transcript_variant",-,-,-,-,-,rs1555915879,MODIFIER,NaN,1,SNV,RPL23AP7,EntrezGene,NaN,YES,NaN,3.982e-06,0,0,0,0,0,8.817e-06,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22:50768826:T:A
191771,22_50768826_T/A,22:50768826,A,ENSG00000079974,ENST00000685352,Transcript,3_prime_UTR_variant,1284,-,-,-,-,rs1555915879,MODIFIER,NaN,-1,SNV,RABL2B,HGNC,HGNC:9800,NaN,ENSP00000509346,3.982e-06,0,0,0,0,0,8.817e-06,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22:50768826:T:A
191772,22_50768826_T/A,22:50768826,A,ENSG00000079974,ENST00000690024,Transcript,3_prime_UTR_variant,1217,-,-,-,-,rs1555915879,MODIFIER,NaN,-1,SNV,RABL2B,HGNC,HGNC:9800,NaN,ENSP00000509957,3.982e-06,0,0,0,0,0,8.817e-06,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22:50768826

In [56]:
len(df_vep["Uploaded_variation"].unique())

17789

In [57]:
df_vep = df_vep[df_vep["Consequence"].apply(lambda x: "missense_variant" in x.split(","))]
df_vep

,Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,Existing_variation,IMPACT,DISTANCE,STRAND,VARIANT_CLASS,SYMBOL,SYMBOL_SOURCE,HGNC_ID,CANONICAL,ENSP,gnomADe_AF,gnomADe_AFR_AF,gnomADe_AMR_AF,gnomADe_ASJ_AF,gnomADe_EAS_AF,gnomADe_FIN_AF,gnomADe_NFE_AF,gnomADe_OTH_AF,gnomADe_SAS_AF,gnomADg_AF,gnomADg_AFR_AF,gnomADg_AMI_AF,gnomADg_AMR_AF,gnomADg_ASJ_AF,gnomADg_EAS_AF,gnomADg_FIN_AF,gnomADg_MID_AF,gnomADg_NFE_AF,gnomADg_OTH_AF,gnomADg_SAS_AF,CLIN_SIG,PHENO,FLAGS,REVEL,am_class,am_pathogenicity,SOMATIC,Variant
2,1_939437_C/T,1:939437,T,ENSG00000187634,ENST00000342066,Transcript,missense_variant,773,683,228,P/L,cCc/cTc,rs201898716,MODERATE,NaN,1,SNV,SAMD11,HGNC,HGNC:28706,NaN,ENSP00000342313,0.0003007,0,0.000449,0.0007165,0,5.035e-05,0.0001117,0.0003389,0.001172,0.0001626,2.625e-05,0,0.000144,0.0006169,0,0.0001157,0,0.0001687,0,0.001397,uncertain_significance,1,NaN,0.080,likely_benign,0.0781,NaN,1:939437:C:T
4,1_939437_C/T,1:939437,T,ENSG00000187634,ENST00000455979,Transcript,missense_variant,163,164,55,P/L,cCc/cTc,rs201898716,MODERATE,NaN,1,SNV,SAMD11,HGNC,HGNC:28706,NaN,ENSP00000412228,0.0003007,0,0.000449,0.0007165,0,5.035e-05,0.0001117,0.0003389,0.001172,0.0001626,2.625e-05,0,0.000144,0.0006169,0,0.0001157,0,0.0001687,0,0.001397,uncertain_significance,1,cds_start_NF,NaN,NaN,NaN,NaN,1:939437:C:T
13,1_939437_C/T,1:939437,T,ENSG00000187634,ENST00000617307,Transcript,missense_variant,686,686,229,P/L,cCc/cTc,rs201898716,MODERATE,NaN,1,SNV,SAMD11,HGNC,HGNC:28706,NaN,ENSP00000482090,0.0003007,0,0.000449,0.0007165,0,5.035e-05,0.0001117,0.0003389,0.001172,0.0001626,2.625e-05,0,0.000144,0.0006169,0,0.0001157,0,0.0001687,0,0.001397,uncertain_significance,1,cds_start_NF,NaN,NaN,NaN,NaN,1:939437:C:T
17,1_939437_C/T,1:939437,T,ENSG00000187634,ENST00000622503,Transcript,missense_variant,686,686,229,P/L,cCc/cTc,rs201898716,MODERATE,NaN,1,SNV,SAMD11,HGNC,HGNC:28706,NaN,ENSP00000482138,0.0003007,0,0.000449,0.0007165,0,5.035e-05,0.0001117,0.0003389,0.001172,0.0001626,2.625e-05,0,0.000144,0.0006169,0,0.0001157,0,0.0001687,0,0.001397,uncertain_significance,1,cds_start_NF,NaN,NaN,NaN,NaN,1:939437:C:T
18,1_946422_C/T,1:946422,T,ENSG00000188976,ENST00000327044,Transcript,missense_variant,1799,1783,595,V/I,Gtc/Atc,rs777256112,MODERATE,NaN,-1,SNV,NOC2L,HGNC,HGNC:24517,YES,ENSP00000317992,1.594e-05,0,2.893e-05,0,0,0,1.763e-05,0,3.266e-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.232,likely_benign,0.0871,NaN,1:946422:C:T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191761,22_50768826_T/A,22:50768826,A,ENSG00000079974,ENST00000395593,Transcript,missense_variant,1021,667,223,S/C,Agc/Tgc,rs1555915879,MODERATE,NaN,-1,SNV,RABL2B,HGNC,HGNC:9800,NaN,ENSP00000378958,3.982e-06,0,0,0,0,0,8.817e-06,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.173,NaN,NaN,NaN,22:50768826:T:A
191762,22_50768826_T/A,22:50768826,A,ENSG00000079974,ENST00000395595,Transcript,missense_variant,1081,640,214,S/C,Agc/Tgc,rs1555915879,MODERATE,NaN,-1,SNV,RABL2B,HGNC,HGNC:9800,NaN,ENSP00000378960,3.982e-06,0,0,0,0,0,8.817e-06,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.173,NaN,NaN,NaN,22:50768826:T:A
191763,22_50768826_T/A,22:50768826,A,ENSG00000079974,ENST00000395598,Transcript,missense_variant,849,637,213,S/C,Agc/Tgc,rs1555915879,MODERATE,NaN,-1,SNV,RABL2B,HGNC,HGNC:9800,NaN,ENSP00000378962,3.982e-06,0,0,0,0,0,8.817e-06,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.173,likely_benign,0.0624,NaN,22:50768826:T:A
191765,22_50768826_T/A,22:50768826,A,ENSG00000079974,ENST00000435118,Transcript,missense_variant,852,637,213,S/C,Agc/Tgc,rs1555915879,MODERATE,NaN,-1,SNV,RABL2B,HGNC,HGNC:9800,NaN,ENSP00000401906,3.982e-06,0,0,0,0,0,8.817e-06,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [58]:
dr = f"{folder_path}data/analysis/Disrupted_PPI/"
df_raw = pd.read_csv(f"{dr}UCSF_Fu_de_novo_variants_hg38.txt", sep = "\t")
df_raw

,Variant,TYPE,SAMPLE,STATUS,Source,MPC
0,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419
1,3:123327758:G:A,SNV,SSC04674,case,UCSF,1.435280
2,7:55964634:C:T,SNV,SSC05819,control,UCSF,0.607774
3,3:8994422:C:T,SNV,SSC12506,case,UCSF,1.520430
4,6:32195810:C:G,SNV,SSC12506,case,UCSF,0.894647
...,...,...,...,...,...,...
19839,19:11723008:G:C,SNV,SP0023172,case,Fu,0.417360
19840,17:34961586:C:T,SNV,SP0022147,case,Fu,0.424820
19841,16:21203187:C:T,SNV,SP0001383,case,Fu,0.067960
19842,7:76425079:G:A,SNV,SP0003489,case,Fu,0.014326


In [59]:
df_annot = pd.merge(df_raw, df_vep[["Variant", "ENSP", "Protein_position", "gnomADg_AF", "REVEL", "am_class", "am_pathogenicity"]].drop_duplicates(), on = "Variant")
df_annot["gnomADg_AF"] = df_annot["gnomADg_AF"].fillna(-np.inf).astype(float)
df_annot["REVEL"] = df_annot["REVEL"].astype(float)
df_annot

,Variant,TYPE,SAMPLE,STATUS,Source,MPC,ENSP,Protein_position,gnomADg_AF,REVEL,am_class,am_pathogenicity
0,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000262915,8,-inf,0.629,likely_pathogenic,0.708
1,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000333494,8,-inf,0.629,likely_pathogenic,0.708
2,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000335633,8,-inf,0.629,likely_pathogenic,0.708
3,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000317192,8,-inf,0.629,likely_pathogenic,0.708
4,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000316999,8,-inf,0.629,likely_pathogenic,0.708
...,...,...,...,...,...,...,...,...,...,...,...,...
81633,7:76425079:G:A,SNV,SP0003489,case,Fu,0.014326,ENSP00000378326,39,0.000178,0.014,likely_benign,0.076
81634,16:3383002:G:T,SNV,SP0021852,case,Fu,0.016812,ENSP00000302502,436,0.000007,0.053,NaN,NaN
81635,16:3383002:G:T,SNV,SP0021852,case,Fu,0.016812,ENSP00000380061,648,0.000007,0.053,likely_pathogenic,0.9284
81636,16:3383002:G:T,SNV,SP0021852,case,Fu,0.016812,ENSP00000391787,359,0.000007,0.053,NaN,NaN


In [60]:
df_annot["am_class"].dropna().shape[0]

47954

In [61]:
df_annot["am_class"].unique()

array(['likely_pathogenic', nan, 'ambiguous', 'likely_benign'],
      dtype=object)

In [62]:
df_annot[df_annot["am_class"]=="likely_pathogenic"].shape[0]

16250

In [63]:
df_annot["gnomADg_AF"].dropna().max()

np.float64(0.3199)

In [64]:
df_annot["STATUS"].unique()

array(['case', 'control'], dtype=object)

In [65]:
df_annot.head()

,Variant,TYPE,SAMPLE,STATUS,Source,MPC,ENSP,Protein_position,gnomADg_AF,REVEL,am_class,am_pathogenicity
0,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000262915,8,-inf,0.629,likely_pathogenic,0.708
1,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000333494,8,-inf,0.629,likely_pathogenic,0.708
2,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000335633,8,-inf,0.629,likely_pathogenic,0.708
3,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000317192,8,-inf,0.629,likely_pathogenic,0.708
4,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000316999,8,-inf,0.629,likely_pathogenic,0.708


In [66]:
metric_list = ["am_class", "REVEL", "MPC"]
for metric in metric_list:
    df_annot[f"damaging_by_{metric}"] = df_annot.apply(lambda x: damaging_flag(x, metric), axis = 1)
    print(f"damaging by {metric}: {df_annot[f'damaging_by_{metric}'].sum()}")
for metric_x, metric_y in combinations(metric_list, 2):
    combined_metric = f"{metric_x.replace('_', '')}_{metric_y.replace('_', '')}"
    df_annot[f"damaging_by_{combined_metric}"] = df_annot.apply(lambda x: damaging_flag(x, f"{metric_x},{metric_y}"), axis = 1)
    print(f"damaging by {combined_metric}: {df_annot[f'damaging_by_{combined_metric}'].sum()}")
combined_metric = "_".join([metric.replace("_", "") for metric in metric_list])
df_annot[f"damaging_by_{combined_metric}"] = df_annot.apply(lambda x: damaging_flag(x, ",".join(metric_list)), axis = 1)
print(f"damaging by {combined_metric}: {df_annot[f'damaging_by_{combined_metric}'].sum()}")

damaging by am_class: 16250
damaging by REVEL: 11967
damaging by MPC: 8700
damaging by amclass_REVEL: 22583
damaging by amclass_MPC: 20659
damaging by REVEL_MPC: 18107
damaging by amclass_REVEL_MPC: 25915


In [67]:
df_annot["rare_by_gnomAD"] = df_annot["gnomADg_AF"] <= 0
df_annot["rare_by_None"] = True
print(f"rare by gnomAD: {df_annot['rare_by_gnomAD'].sum()}")    
print(f"rare by None: {df_annot['rare_by_None'].sum()}")

rare by gnomAD: 58221
rare by None: 81638


In [ ]:
dr = f"{folder_path}data/analysis/Disrupted_PPI/"
df_annot.to_csv(f"{dr}UCSF_Fu_de_novo_variants_hg38.full.txt", index = False, sep = "\t")

# Map disrupted PPIs

In [68]:
dr = f"{folder_path}src/NetFlow3D/metadata/"
df_ppi_interface = pd.read_csv(dr + "HomoSapiens_interfaces_PIONEER_veryhigh.txt", sep="\t")
df_ppi_interface

,P1,P2,P1_IRES,P2_IRES,Interaction,Source
0,P28062,P40306,"P28062_269,P28062_272,P28062_273",P40306_251,"P28062,P40306",PDB
1,O14977,O14977,"O14977_158,O14977_174,O14977_175,O14977_204,O1...","O14977_158,O14977_174,O14977_175,O14977_204,O1...","O14977,O14977",PDB
2,Q7Z7F7,Q9H2W6,"Q7Z7F7_34,Q7Z7F7_35,Q7Z7F7_36,Q7Z7F7_37,Q7Z7F7...","Q9H2W6_57,Q9H2W6_58,Q9H2W6_59,Q9H2W6_60,Q9H2W6...","Q7Z7F7,Q9H2W6",PDB
3,P62263,P62857,"P62263_61,P62263_70,P62263_71,P62263_72,P62263...","P62857_5,P62857_6,P62857_7,P62857_8,P62857_10,...","P62263,P62857",PDB
4,P08603,P08603,"P08603_241,P08603_265,P08603_336,P08603_337,P0...","P08603_241,P08603_265,P08603_336,P08603_337,P0...","P08603,P08603",PDB
...,...,...,...,...,...,...
146132,V9HW27,V9HW27,NaN,NaN,"V9HW27,V9HW27",PIONEER
146133,V9HW80,V9HW80,NaN,NaN,"V9HW80,V9HW80",PIONEER
146134,V9HW89,V9HW89,NaN,NaN,"V9HW89,V9HW89",PIONEER
146135,V9HWJ5,V9HWJ5,NaN,NaN,"V9HWJ5,V9HWJ5",PIONEER


In [69]:
dr = f"{folder_path}data/analysis/Disrupted_PPI/"
df_annot = pd.read_csv(f"{dr}UCSF_Fu_de_novo_variants_hg38.full.txt", sep="\t")
df_annot = df_annot[df_annot["STATUS"] == "case"]
df_annot

,Variant,TYPE,SAMPLE,STATUS,Source,MPC,ENSP,Protein_position,gnomADg_AF,REVEL,am_class,am_pathogenicity,damaging_by_am_class,damaging_by_REVEL,damaging_by_MPC,damaging_by_amclass_REVEL,damaging_by_amclass_MPC,damaging_by_REVEL_MPC,damaging_by_amclass_REVEL_MPC,rare_by_gnomAD,rare_by_None
0,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000262915,8,-inf,0.629,likely_pathogenic,0.7080,True,True,False,True,True,True,True,True,True
1,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000333494,8,-inf,0.629,likely_pathogenic,0.7080,True,True,False,True,True,True,True,True,True
2,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000335633,8,-inf,0.629,likely_pathogenic,0.7080,True,True,False,True,True,True,True,True,True
3,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000317192,8,-inf,0.629,likely_pathogenic,0.7080,True,True,False,True,True,True,True,True,True
4,1:43736284:C:T,SNV,SSC04674,case,UCSF,0.780419,ENSP00000316999,8,-inf,0.629,likely_pathogenic,0.7080,True,True,False,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81633,7:76425079:G:A,SNV,SP0003489,case,Fu,0.014326,ENSP00000378326,39,0.000178,0.014,likely_benign,0.0760,False,False,False,False,False,False,False,False,True
81634,16:3383002:G:T,SNV,SP0021852,case,Fu,0.016812,ENSP00000302502,436,0.000007,0.053,NaN,NaN,False,False,False,False,False,False,False,False,True
81635,16:3383002:G:T,SNV,SP0021852,case,Fu,0.016812,ENSP00000380061,648,0.000007,0.053,likely_pathogenic,0.9284,True,False,False,True,True,False,True,False,True
81636,16:3383002:G:T,SNV,SP0021852,case,Fu,0.016812,ENSP00000391787,359,0.000007,0.053,NaN,NaN,False,False,False,False,False,False,False,False,True


In [70]:
rare_cols = [c for c in df_annot.columns if c.startswith("rare_by_")]
damaging_cols = [c for c in df_annot.columns if c.startswith("damaging_by_")]
print("rare cols:", rare_cols)
print("damaging cols:", damaging_cols)

rare cols: ['rare_by_gnomAD', 'rare_by_None']
damaging cols: ['damaging_by_am_class', 'damaging_by_REVEL', 'damaging_by_MPC', 'damaging_by_amclass_REVEL', 'damaging_by_amclass_MPC', 'damaging_by_REVEL_MPC', 'damaging_by_amclass_REVEL_MPC']


In [71]:
df_enspmap.head()

,UniProt,ID_type,ID
67,P31946,Ensembl_PRO,ENSP00000300161
69,P31946,Ensembl_PRO,ENSP00000361930
191,P62258,Ensembl_PRO,ENSP00000264335
193,P62258,Ensembl_PRO,ENSP00000461762
196,P62258,Ensembl_PRO,ENSP00000481059


In [ ]:
dr = f"{folder_path}data/analysis/Disrupted_PPI/"
for rare_col in rare_cols:
    for damaging_col in damaging_cols:
        df_tmp = df_annot[df_annot[rare_col] & df_annot[damaging_col] & df_annot["source_nozhou"]]
        print(f"############ {rare_col} & {damaging_col}: {df_tmp.shape[0]}")
        df_tmp = pd.merge(df_tmp[["ENSP", "Protein_position"]].drop_duplicates(), df_enspmap.drop(columns = ["ID_type"]).drop_duplicates(), left_on="ENSP", right_on="ID")
        display(df_tmp.head())
        mut_res_list = set(df_tmp.apply(lambda x: x["UniProt"] + "_" + str(x["Protein_position"]), axis=1))
        print(f"mut_res_list: {len(mut_res_list)}, {list(mut_res_list)[:10]}")
        df_disrupt_ppi = df_ppi_interface[df_ppi_interface.apply(lambda x: get_disrupted_ppi(x, mut_res_list), axis=1)]
        df_disrupt_ppi = df_disrupt_ppi[["P1", "P2"]].drop_duplicates()
        print(f"disrupted PPIs: {df_disrupt_ppi.shape[0]}")
        tag = f"{rare_col.replace('rare_by_', '')}.{damaging_col.replace('damaging_by_', '')}"
        df_disrupt_ppi.to_csv(dr + f"Disrupted_PPI_{tag}.txt", sep="\t", index=False, header=True)


############ rare_by_gnomAD & damaging_by_am_class: 10693


,ENSP,Protein_position,UniProt,ID
0,ENSP00000262915,8,Q11203,ENSP00000262915
1,ENSP00000333494,8,Q11203,ENSP00000333494
2,ENSP00000335633,8,Q11203,ENSP00000335633
3,ENSP00000317192,8,Q11203,ENSP00000317192
4,ENSP00000316999,8,Q11203,ENSP00000316999


mut_res_list: 5996, ['Q86UA6_145', 'Q6GQQ9_468', 'Q15459_179', 'Q8TAE7_81', 'H3BTW5_855', 'Q5JVZ4_59', 'C9J4Q3_102', 'A0A494C1Q1_46', 'Q15629_242', 'Q9H159_45']
disrupted PPIs: 786
############ rare_by_gnomAD & damaging_by_REVEL: 7001


,ENSP,Protein_position,UniProt,ID
0,ENSP00000262915,8,Q11203,ENSP00000262915
1,ENSP00000333494,8,Q11203,ENSP00000333494
2,ENSP00000335633,8,Q11203,ENSP00000335633
3,ENSP00000317192,8,Q11203,ENSP00000317192
4,ENSP00000316999,8,Q11203,ENSP00000316999


mut_res_list: 4696, ['Q86UA6_145', 'P41743_151', 'Q14194_524', 'O75061_585', 'Q9NS40_403', 'O95985_666', 'F8W7U8_63', 'P12821_363', 'E9PRY3_6', 'Q2TAC6_572']
disrupted PPIs: 663
############ rare_by_gnomAD & damaging_by_MPC: 6856


,ENSP,Protein_position,UniProt,ID
0,ENSP00000261793,183,Q9UQM7,ENSP00000261793
1,ENSP00000381412,183,A0A5K1VW76,ENSP00000381412
2,ENSP00000426607,183,D6RFJ0,ENSP00000426607
3,ENSP00000500386,183,Q9UQM7,ENSP00000500386
4,ENSP00000500700,183,A0A5F9ZHY2,ENSP00000500700


... (19 outputs left out)

,ENSP,Protein_position,UniProt,ID
0,ENSP00000262915,8,Q11203,ENSP00000262915
1,ENSP00000333494,8,Q11203,ENSP00000333494
2,ENSP00000335633,8,Q11203,ENSP00000335633
3,ENSP00000317192,8,Q11203,ENSP00000317192
4,ENSP00000316999,8,Q11203,ENSP00000316999


mut_res_list: 9325, ['P05107_584', 'A0A7P0T8W1_2064', 'Q86UA6_145', 'Q9NS40_403', 'F5GWX5_864', 'C9J660_5', 'Q15459_179', 'Q9NWK9_311', 'P22607_382', 'Q8TAE7_81']
disrupted PPIs: 1013
############ rare_by_None & damaging_by_amclass_REVEL_MPC: 20028


,ENSP,Protein_position,UniProt,ID
0,ENSP00000262915,8,Q11203,ENSP00000262915
1,ENSP00000333494,8,Q11203,ENSP00000333494
2,ENSP00000335633,8,Q11203,ENSP00000335633
3,ENSP00000317192,8,Q11203,ENSP00000317192
4,ENSP00000316999,8,Q11203,ENSP00000316999


mut_res_list: 12531, ['P05107_584', 'A0A7P0T8W1_2064', 'Q86UA6_145', 'Q9NS40_403', 'O43150_526', 'Q6GQQ9_468', 'F5GWX5_864', 'C9J660_5', 'Q15459_179', 'Q9NWK9_311']
disrupted PPIs: 1322
